In [1]:
%load_ext autoreload
%autoreload 2 

: 

In [3]:
from pathlib import Path
import os
import socket

In [4]:
try:
    project_root = next(p for p in Path().cwd().resolve().parents if (p /"env.env").exists())
except StopIteration:
    # Fallback to current working directory if not found
    project_root = Path.cwd()
print(f"Project Root: {project_root}")
ROOT_DATASET_DIR = project_root / "datasets" 

In [5]:
# Check if running on the Pace server
hostname = socket.gethostname()
IS_PACE_SERVER = "pace" in hostname.lower()  # Replace 'pace' with a unique keyword from your server's hostname
IS_PACE_SERVER

In [6]:
import torch
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm import tqdm

In [7]:
_ = torch.manual_seed(0)

In [8]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv( Path.cwd() / "env.env")
PATH_TO_DATA_DIR = Path.cwd() / "dataset"
PATH_TO_DATA_DIR

### Step 1: Load pretrained GPT-2

In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [10]:
model_id = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)
print(model.config.to_dict())

In [11]:
print(f"Total parameters: {sum(parameter.numel() for parameter in model.parameters()):,}")

In [12]:
sample = "Hello, who are you ? "
out = model.generate(tokenizer.encode(sample, return_tensors="pt"), max_new_tokens = 20)
print(tokenizer.decode(out[0], skip_special_tokens=True))

#### Step 2 — Inspect GPT-2 Architecture

In [13]:
total_params = 0

for name, parameter in model.named_parameters():
    parameter_count = parameter.numel()
    total_params += parameter_count

    print(
        f"{name:<60} "
        f"shape={tuple(parameter.shape)!s:<20} "
        f"parameters={parameter_count:,} "
        f"trainable={parameter.requires_grad}"
    )

print(f"\nTotal parameters: {total_params:,}")

In [14]:
for name, module in model.named_modules():
    # print(module.__dict__.keys())
    print(f"{name:<50} {type(module).__name__}  " )

### Step 3 — Load and Inspect the Instruction Dataset

In [16]:
if IS_PACE_SERVER:
    from model_training.post_training.artifacts.data.instruction_tuning.raw_data.simple_example.instruct_data import download_and_load_file
    from model_training.post_training.artifacts.data.instruction_tuning.prompt_style.styles import AlpacaFormat
    from model_training.post_training.artifacts.data.instruction_tuning.dataloader import InstructionDataset, custom_collate_fn
else:
    ## Fix this import
    raise Exception("Not Running on Pace")
        # from src.model import GPTdataset['train']

In [ ]:
from datasets import load_dataset, DatasetDict

data = load_dataset("yahma/alpaca-cleaned", split="train")
print(data)

In [ ]:
alpaca = AlpacaFormat(data)
## Example of format
print(alpaca.format_input(instruction=data[50]['instruction'], in_text=data[50]['input']))
print("=============================================")
print(alpaca.detailed_response(out_text = data[50]['output']))

In [ ]:
sample = (alpaca.complete_response(instruction=data[999]['instruction'], in_text=data[999]['input'], out_text=data[999]['output']))
print(sample)

#### Spliting the data 

In [ ]:
# 1. Split the original dataset into 80% train and 20% remainder (test + valid)
train_remainder = data.train_test_split(test_size=0.2, seed=42)

# 2. Split the 20% remainder in half to get 10% test and 10% validation
test_valid = train_remainder["test"].train_test_split(test_size=0.5, seed=42)

# 3. Combine everything into a clean DatasetDict
final_dataset = DatasetDict({
    "train": train_remainder["train"],
    "validation": test_valid["train"],  # The 'train' side of the remainder split
    "test": test_valid["test"]          # The 'test' side of the remainder split
})

In [ ]:
dataloader = InstructionDataset(ds=final_dataset['train'], tokenizer=tokenizer, prompt_style=alpaca)
(sample)